# Deutsch-Jozsa and Bernstein-Vazirani on real hardware

| | |
|---|---|
| **Level** | Introductory to intermediate |
| **Time** | About 45 minutes |
| **Prerequisites** | Hadamard and CNOT gates. Phase kickback is explained below. |
| **Default device** | Rigetti Cepheus-1-108Q |
| **Also runs on** | IQM Garnet, AQT IBEX Q1 (in its scheduled windows) |
| **Qubits** | 4 |
| **Two-qubit gates** | 2 to 3 per circuit, before routing |
| **Hardware jobs** | 3 |
| **Approximate cost** | Rigetti: about 30 credits in total (billed by execution time). Garnet at 1000 shots: about 525 credits. |
| **Suggested hand-in** | Your results table, and answers to Questions 1 to 3 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

*Part of the QUEST notebooks from qBraid. You may copy, edit and adapt this notebook for your course.*

Both algorithms answer a question about a hidden function $f$ with a single query, where a classical computer needs several.

- **Deutsch-Jozsa.** $f$ takes $n$ bits to one bit, and is promised to be either *constant* (the same output for every input) or *balanced* (0 for half the inputs, 1 for the other half). Which is it? Classically, you may need $2^{n-1} + 1$ queries to be sure. The quantum circuit needs one. It outputs all zeros if $f$ is constant, and anything else if $f$ is balanced.
- **Bernstein-Vazirani.** $f(x) = s \cdot x \bmod 2$ for a hidden bit string $s$. Classically you need $n$ queries to find $s$. The quantum circuit returns $s$ directly.

Both use **phase kickback**. An extra qubit, the *ancilla*, is prepared in $|-\rangle$. When the oracle applies a CNOT onto it, the ancilla is unchanged, but the control qubit picks up a sign of $-1$. The final Hadamards turn those signs into bits you can measure.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "rigetti:rigetti:qpu:cepheus-1-108q"   # device list and prices: see the README
SHOTS = 1000                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits
HW_BASIS = ['rz', 'rx', 'ry', 'cz', 'cx', 'h', 'measure']   # gates every QUEST device accepts
QUEST_JOB_TAGS = {"quest": "algo-djbv"}   # labels this notebook's hardware jobs for QUEST usage statistics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

## 1. Build the circuits

Three circuits: Deutsch-Jozsa with a constant function, Deutsch-Jozsa with a balanced function (the parity of the input bits), and Bernstein-Vazirani with the secret string `SECRET`.

In [ ]:
SECRET = "101"      # hidden string for Bernstein-Vazirani. Qubit 0 is the rightmost bit.

def query_circuit(n_inputs, oracle):
    """H on everything, one oracle call, H on the inputs, measure the inputs."""
    ancilla = n_inputs
    qc = QuantumCircuit(n_inputs + 1, n_inputs)
    qc.x(ancilla)
    qc.h(range(n_inputs + 1))        # inputs in superposition, ancilla in |->
    oracle(qc, ancilla)
    qc.h(range(n_inputs))
    qc.measure(range(n_inputs), range(n_inputs))
    return qc

def constant_oracle(qc, ancilla):
    pass                             # f(x) = 0 for every x

def parity_oracle(qc, ancilla):
    for q in range(ancilla):         # f(x) = x0 xor x1 xor x2
        qc.cx(q, ancilla)

def secret_oracle(qc, ancilla):
    for q, bit in enumerate(reversed(SECRET)):
        if bit == "1":               # f(x) = s . x mod 2
            qc.cx(q, ancilla)

circuits = {
    "DJ, constant f": query_circuit(3, constant_oracle),
    "DJ, balanced f": query_circuit(3, parity_oracle),
    f"BV, secret {SECRET}": query_circuit(len(SECRET), secret_oracle),
}
expected = {"DJ, constant f": "000", "DJ, balanced f": "111", f"BV, secret {SECRET}": SECRET}

circuits[f"BV, secret {SECRET}"].draw(output="text")

## 2. Ideal simulation

Each circuit gives its expected answer on every shot.

In [ ]:
simulator = AerSimulator()
ideal = {}
for name, qc in circuits.items():
    ideal[name] = simulator.run(qc, shots=SHOTS).result().get_counts()
    print(f"{name:18s} expected {expected[name]}   got {ideal[name]}")

## 3. Run on hardware

Before submitting, each circuit is converted to gates that every QUEST device accepts. The printed counts are two-qubit gates before routing. On devices where qubits only connect to their neighbours, routing can add more.

The constant-function circuit shows 0 two-qubit gates: its Hadamard gates cancel in pairs, and the compiler removes them. On hardware it therefore measures readout error and nothing else, which makes it a useful reference for the other two.

In [ ]:
hw_circuits = {name: transpile(qc, basis_gates=HW_BASIS, optimization_level=1)
               for name, qc in circuits.items()}
for name, qc in hw_circuits.items():
    ops = qc.count_ops()
    print(f"{name:18s} two-qubit gates: {ops.get('cx', 0) + ops.get('cz', 0)}")

In [ ]:
N_JOBS = len(hw_circuits)

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    jobs = {name: device.run(qc, shots=SHOTS, tags=QUEST_JOB_TAGS) for name, qc in hw_circuits.items()}
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = {name: job.result().data.get_counts() for name, job in jobs.items()}
    print("Received results for", list(hw_counts))
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 4. Compare

For each circuit, the success rate is the fraction of shots that returned the expected answer.

In [ ]:
def success(counts, answer):
    total = sum(counts.values())
    return sum(n for key, n in counts.items() if key.replace(" ", "").zfill(len(answer)) == answer) / total

names = list(circuits)
ideal_rates = [success(ideal[n], expected[n]) for n in names]
x = np.arange(len(names))
plt.bar(x - 0.2, ideal_rates, width=0.4, color="gray", label="ideal simulation")

if hw_counts:
    hw_rates = [success(hw_counts[n], expected[n]) for n in names]
    plt.bar(x + 0.2, hw_rates, width=0.4, color="tab:orange", label=f"hardware ({DEVICE_ID})")
    for n in names:
        top = sorted(hw_counts[n].items(), key=lambda kv: -kv[1])[:4]
        print(f"{n:18s} success {success(hw_counts[n], expected[n]):.3f}   most common: {top}")

plt.axhline(1 / 8, color="black", linestyle=":", label="random guessing (1/8)")
plt.xticks(x, names)
plt.ylabel("fraction of shots with the expected answer")
plt.legend()
plt.show()

## Questions to try

1. Classically, how many evaluations of a 3-bit function do you need, in the worst case, to be certain it is constant or balanced? How many does the quantum circuit use?
2. Look at the wrong answers on hardware. Do they mostly differ from the correct answer by one bit, or by several? Which kind of noise from [the noise notebook](../noise_and_hardware/intro_01_noise_on_real_hardware.ipynb) does that suggest?
3. Which of the three circuits did best on hardware, and which did worst? Relate your answer to the number of two-qubit gates in each.
4. Set `SECRET = "1011"`, which needs four input qubits. How does the success rate change as the secret gets longer?
5. Write a different balanced oracle, for example $f(x) = x_0$. What output does Deutsch-Jozsa give?